
# Assignment 2: Exploring Text Generation Using GPT-2

## Objective

The objective of this assignment is to understand how different text-generation parameters influence the output produced by a pretrained language model.

In this experiment, the **GPT-2** model from Hugging Face Transformers is used. A fixed opening sentence is passed to the model multiple times while changing one generation parameter at a time.

The experiments focus on:

- Temperature
- Top-k
- Top-p
- Repetition penalty
- A high-temperature setting to observe unusual or incoherent output


## 1. Install and Import Required Libraries

In [4]:

pip install transformers


  Using cached transformers-5.16.1-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.30.0-py3-none-any.whl.metadata (16 kB)
  Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached transformers-5.16.1-py3-none-any.whl (12.1 MB)
Using cached huggingface_hub-1.30.0-py3-none-any.whl (796 kB)
Using cached pyyaml-6.0.3-cp311-cp311-win_amd64.whl (158 kB)
Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl (355 kB)
Using cached tokenizers-0.23.2-cp310-abi3-win_amd64.whl (2.9 MB)
Using cached click-8.5.0-py3-none-any.whl (125 kB)
Using cached hf_xet-1.6.0-cp38-abi3-win_amd64.whl (4.0 MB)
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Un


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:

import torch
import pandas as pd

from transformers import GPT2LMHeadModel, GPT2Tokenizer, set_seed

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


c:\Users\vishe\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.14.0+cpu
CUDA available: False



## 2. Load the Pretrained GPT-2 Model

GPT-2 is a pretrained autoregressive language model. It generates text one token at a time based on the text that came before it.


In [6]:

model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 does not have a separate padding token by default.
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

print("Model loaded successfully.")
print("Device:", device)


c:\Users\vishe\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vishe\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 10365.99it/s]


Model loaded successfully.
Device: cpu



## 3. Define the Original Prompt

The same prompt is used for the five main experiments so that the effect of the generation parameters can be compared fairly.

### Original prompt

> **The old house at the end of the street had been empty for years, until one rainy evening**


In [7]:

original_prompt = (
    "The old house at the end of the street had been empty for years, "
    "until one rainy evening"
)

print(original_prompt)


The old house at the end of the street had been empty for years, until one rainy evening



## 4. Text Generation Function

Sampling is enabled so that temperature, top-k, and top-p can influence the generated text.

A fixed seed is used for reproducibility. The model, prompt, maximum output length, and other baseline settings remain constant between experiments unless the experiment specifically changes that parameter.


In [8]:

def generate_text(
    prompt,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    seed=42,
    max_new_tokens=100
):
    set_seed(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return generated_text



# 5. Five Parameter Experiments

For the main comparison, the baseline values are:

- Temperature = 1.0
- Top-k = 50
- Top-p = 1.0
- Repetition penalty = 1.0

Each experiment changes only one parameter.


### Experiment 1 — Low Temperature

A low temperature makes the model favor higher-probability tokens, generally producing more predictable and conservative text.

In [9]:

exp1 = generate_text(
    original_prompt,
    temperature=0.3,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    seed=42
)

print("Parameter changed: Temperature")
print("Value: 0.3")
print("\nGenerated output:\n")
print(exp1)


Parameter changed: Temperature
Value: 0.3

Generated output:

The old house at the end of the street had been empty for years, until one rainy evening, when the landlord came to see what was going on. He had been looking for a place to stay, and when he saw the old house, he ran to it and found that it was empty. He then went to the old house, and found that the old house was full of people. He then went to the old house, and found that the old house was full of people. He then went to the old house, and found that the old house was full of people. He then


### Experiment 2 — High Temperature

A high temperature gives lower-probability tokens a greater chance of being selected, increasing creativity and randomness.

In [10]:

exp2 = generate_text(
    original_prompt,
    temperature=1.5,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    seed=42
)

print("Parameter changed: Temperature")
print("Value: 1.5")
print("\nGenerated output:\n")
print(exp2)


Parameter changed: Temperature
Value: 1.5

Generated output:

The old house at the end of the street had been empty for years, until one rainy evening when somebody knocked a door at three in the evening . And the police had a new story to tell. Mr Leach found the young young man lying dead and buried in several rooms. They had been sleeping when his aunt noticed him. The body had long been moved after death by the man who gave it so strong the officers called by law enforcement found him dead on foot. He was carried in handcuffs to the court, where by law police tried almost invariably to seize one of the two young man's


### Experiment 3 — Low Top-k

Top-k limits sampling to the k most likely next tokens. A smaller value restricts the model's choices and can make the output more focused.

In [11]:

exp3 = generate_text(
    original_prompt,
    temperature=1.0,
    top_k=10,
    top_p=1.0,
    repetition_penalty=1.0,
    seed=42
)

print("Parameter changed: Top-k")
print("Value: 10")
print("\nGenerated output:\n")
print(exp3)


Parameter changed: Top-k
Value: 10

Generated output:

The old house at the end of the street had been empty for years, until one rainy evening when a group of teenagers came in and started beating up the neighbors. The group took out a few guns, but they also started shooting at the neighborhood's children, who were playing. They had been playing outside for some time, but suddenly the neighborhood started being attacked by the teenagers. As the teenagers ran off, the group started shooting at the children, who started shooting at them. The teens got out guns and started shooting at the neighborhood again. The teen who killed the teenagers was later named as


### Experiment 4 — Low Top-p

Top-p, also called nucleus sampling, selects tokens from the smallest group whose cumulative probability reaches p. A lower value restricts the generation to a narrower probability mass.

In [12]:

exp4 = generate_text(
    original_prompt,
    temperature=1.0,
    top_k=50,
    top_p=0.7,
    repetition_penalty=1.0,
    seed=42
)

print("Parameter changed: Top-p")
print("Value: 0.7")
print("\nGenerated output:\n")
print(exp4)


Parameter changed: Top-p
Value: 0.7

Generated output:

The old house at the end of the street had been empty for years, until one rainy evening when a group of teenagers came in and started beating up the neighbors. The group eventually stopped, and it turned out that the boys had been arrested for robbery and vandalism. They were charged with felony burglary.

This was the beginning of the end for the group, as they started to show signs of sobriety.

The group, which had been known as the Catherines, was soon arrested, but by the time they were released they were in the process of getting their house


### Experiment 5 — Repetition Penalty

A repetition penalty discourages the model from repeatedly selecting tokens that have already appeared. A value above 1.0 can reduce repetitive phrases.

In [13]:

exp5 = generate_text(
    original_prompt,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.3,
    seed=42
)

print("Parameter changed: Repetition penalty")
print("Value: 1.3")
print("\nGenerated output:\n")
print(exp5)


Parameter changed: Repetition penalty
Value: 1.3

Generated output:

The old house at the end of the street had been empty for years, until one rainy evening when somebody knocked on it. It was like something out there in some dark attic bedroom."
Ranel says they found a large stone slab lying next to what looked less than three feet below ground level beneath his own backyard patio room — which overlooks another alleyway that is connected by brick undergrowth surrounding former homes south Woonsocket Avenue or two-plus foot streets (or more). They say he did not know why this piece of soil may have fallen so quickly away after all their digging


## 6. Five-Experiment Comparison Table

In [14]:

experiment_results = pd.DataFrame({
    "Experiment": [
        "1. Low Temperature",
        "2. High Temperature",
        "3. Low Top-k",
        "4. Low Top-p",
        "5. Repetition Penalty"
    ],
    "Parameter Changed": [
        "Temperature",
        "Temperature",
        "Top-k",
        "Top-p",
        "Repetition penalty"
    ],
    "Value": [
        0.3,
        1.5,
        10,
        0.7,
        1.3
    ],
    "Generated Output": [
        exp1,
        exp2,
        exp3,
        exp4,
        exp5
    ]
})

pd.set_option("display.max_colwidth", 500)
display(experiment_results)


,Experiment,Parameter Changed,Value,Generated Output
0,1. Low Temperature,Temperature,0.3,"The old house at the end of the street had been empty for years, until one rainy evening, when the landlord came to see what was going on. He had been looking for a place to stay, and when he saw the old house, he ran to it and found that it was empty. He then went to the old house, and found that the old house was full of people. He then went to the old house, and found that the old house was full of people. He then went to the old house, and found that the old house was full of people. He ..."
1,2. High Temperature,Temperature,1.5,"The old house at the end of the street had been empty for years, until one rainy evening when somebody knocked a door at three in the evening . And the police had a new story to tell. Mr Leach found the young young man lying dead and buried in several rooms. They had been sleeping when his aunt noticed him. The body had long been moved after death by the man who gave it so strong the officers called by law enforcement found him dead on foot. He was carried in handcuffs to the court, where by..."
2,3. Low Top-k,Top-k,10.0,"The old house at the end of the street had been empty for years, until one rainy evening when a group of teenagers came in and started beating up the neighbors. The group took out a few guns, but they also started shooting at the neighborhood's children, who were playing. They had been playing outside for some time, but suddenly the neighborhood started being attacked by the teenagers. As the teenagers ran off, the group started shooting at the children, who started shooting at them. The tee..."
3,4. Low Top-p,Top-p,0.7,"The old house at the end of the street had been empty for years, until one rainy evening when a group of teenagers came in and started beating up the neighbors. The group eventually stopped, and it turned out that the boys had been arrested for robbery and vandalism. They were charged with felony burglary.\n\nThis was the beginning of the end for the group, as they started to show signs of sobriety.\n\nThe group, which had been known as the Catherines, was soon arrested, but by the time they..."
4,5. Repetition Penalty,Repetition penalty,1.3,"The old house at the end of the street had been empty for years, until one rainy evening when somebody knocked on it. It was like something out there in some dark attic bedroom.""\nRanel says they found a large stone slab lying next to what looked less than three feet below ground level beneath his own backyard patio room — which overlooks another alleyway that is connected by brick undergrowth surrounding former homes south Woonsocket Avenue or two-plus foot streets (or more). They say he di..."



## 7. Observations

### Experiment 1 — Temperature = 0.3
The low temperature setting should produce relatively predictable and focused text because the model strongly favors high-probability tokens.

### Experiment 2 — Temperature = 1.5
The high temperature setting should produce more varied and creative text. However, increasing randomness too much can cause the text to become less coherent or logically inconsistent.

### Experiment 3 — Top-k = 10
The model is restricted to only the 10 most likely candidate tokens at each generation step. This reduces the range of possible choices and generally makes the output more controlled.

### Experiment 4 — Top-p = 0.7
The model samples from a relatively narrow probability mass. This normally keeps the text focused while still allowing some variation.

### Experiment 5 — Repetition penalty = 1.3
The model is discouraged from repeating previously generated tokens. This can reduce repetitive wording, although an excessively large penalty can sometimes make the text unnatural.



# 8. Additional Activity — Very High Temperature

A very high temperature is included to demonstrate why excessive randomness can produce unusual or meaningless text.

**Important:** The exact output will vary because GPT-2 is probabilistic. If the result is particularly unusual, keep the output shown below and take a screenshot for submission.


In [15]:

high_temp_output = generate_text(
    original_prompt,
    temperature=2.0,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    seed=123
)

print("Parameter changed: Temperature")
print("Value: 2.0")
print("\nHigh-temperature output:\n")
print(high_temp_output)


Parameter changed: Temperature
Value: 2.0

High-temperature output:

The old house at the end of the street had been empty for years, until one rainy evening, shortly after I was elected governor. The roof had gone down in disgrace at once; the walls were ripped apart after one storm, then broken as one took breath in its heat that they too found their footing and stood over fallen timber. The trees I was watching twined on fire, as they went forward of its face under each brush of raindrops and through those long lines of snow from down to the roof of my home's garden all with what remained of its ashes on a single flat wall



### Why can high temperature produce unusual output?

Temperature changes how strongly the model favors high-probability tokens. At a very high temperature, lower-probability tokens become much more likely to be selected. This increases randomness and diversity, but it also makes the model more likely to choose words that do not fit the surrounding context. As a result, the generated text can become strange, inconsistent, or partially meaningless.



# 9. Custom Prompt

The following custom prompt is used to generate three different versions.

### Custom prompt

> **Our startup had one week to prove that its new AI assistant could save customers time.**


In [16]:

custom_prompt = (
    "Our startup had one week to prove that its new AI assistant "
    "could save customers time."
)

print(custom_prompt)


Our startup had one week to prove that its new AI assistant could save customers time.


## Custom Generation 1 — Conservative

In [17]:

custom_1 = generate_text(
    custom_prompt,
    temperature=0.5,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.1,
    seed=10
)

print(custom_1)


Our startup had one week to prove that its new AI assistant could save customers time.



## Custom Generation 2 — Balanced

In [18]:

custom_2 = generate_text(
    custom_prompt,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
    seed=20
)

print(custom_2)


Our startup had one week to prove that its new AI assistant could save customers time.
, as an individual who works at a hardware supplier and has never seen how it performs on human eyes before (which they have since learned is accurate); if we really wanted this "bot" from Apple's App Store product repository for us to use our hands in the app store where people purchase software products rather than just buy something else; or when those companies are getting more creative with their own SDKs but I guess having them help create apps based off of Android does NOT make sense because these startups


## Custom Generation 3 — Creative

In [19]:

custom_3 = generate_text(
    custom_prompt,
    temperature=1.3,
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.1,
    seed=30
)

print(custom_3)


Our startup had one week to prove that its new AI assistant could save customers time. And before I got too excited in our interview, let's consider what the real value of a smart budget list is:
- It should work better when you set it up with your best friends instead on Craigslist or Yahoo Answers (or whatever app would have already managed them!). - You can still show people how much information about anyone nearby isn't coming from outside sources as quickly and easily as just saying good morning through an iOS search engine where no human involved has actual knowledge whatsoever! For some companies at


## 10. Custom Generation Comparison

In [20]:

custom_results = pd.DataFrame({
    "Version": ["Version 1", "Version 2", "Version 3"],
    "Temperature": [0.5, 0.9, 1.3],
    "Top-k": [40, 50, 50],
    "Top-p": [0.9, 0.95, 0.95],
    "Generated Output": [custom_1, custom_2, custom_3]
})

display(custom_results)


,Version,Temperature,Top-k,Top-p,Generated Output
0,Version 1,0.5,40,0.90,Our startup had one week to prove that its new AI assistant could save customers time.\n
1,Version 2,0.9,50,0.95,"Our startup had one week to prove that its new AI assistant could save customers time.\n, as an individual who works at a hardware supplier and has never seen how it performs on human eyes before (which they have since learned is accurate); if we really wanted this ""bot"" from Apple's App Store product repository for us to use our hands in the app store where people purchase software products rather than just buy something else; or when those companies are getting more creative with their own..."
2,Version 3,1.3,50,0.95,"Our startup had one week to prove that its new AI assistant could save customers time. And before I got too excited in our interview, let's consider what the real value of a smart budget list is:\n- It should work better when you set it up with your best friends instead on Craigslist or Yahoo Answers (or whatever app would have already managed them!). - You can still show people how much information about anyone nearby isn't coming from outside sources as quickly and easily as just saying go..."



# 11. Final Analysis

The experiments show that generation parameters control the balance between **predictability, diversity, and coherence**. A lower temperature generally produces safer and more predictable text because the model favors the most probable tokens. Higher temperature increases creativity and randomness, but excessive values can reduce coherence. Lower top-k and top-p values restrict the possible next tokens and therefore keep generation more focused. A repetition penalty helps reduce repeated words and phrases. Overall, these parameters do not change what GPT-2 learned during training; they change **how the model samples from the probability distribution it produces**.



# 12. Conclusion

GPT-2 does not simply produce one fixed continuation for a prompt. Its output can change substantially depending on how tokens are sampled from the model's predicted probability distribution. Temperature controls randomness, top-k and top-p control the candidate token set, and repetition penalty discourages repeated content. The experiments demonstrate that generation quality is a trade-off: more restriction generally improves predictability, while more randomness can increase creativity but may also reduce coherence.
